
# RAGES Analysis (Interactive)

This notebook loads `analysis_rages.jsonl`, prints the requested case lists, and provides interactive plots for pipelines A-D.


In [1]:

from __future__ import annotations

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except Exception:
    widgets = None
    display = None
    HAS_WIDGETS = False


def find_root_path(path: str, word: str) -> str:
    parts = path.split(word, 1)
    return parts[0] + word if len(parts) > 1 else path


ROOT_FOLDER = Path(find_root_path(os.getcwd(), 'art_lang'))


In [ ]:

JSONL_PATH = ROOT_FOLDER / 'rpod/rages/out/analysis_rages_test_v2.jsonl'

if not JSONL_PATH.exists():
    raise FileNotFoundError(f'Missing file: {JSONL_PATH}')

records = []
with open(JSONL_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        records.append(json.loads(line))

scenario_rows = []
pipeline_rows = []

for rec in records:
    sid = int(rec.get('scenario_id', -1))
    scp4 = rec.get('scp_convergence_4') or {}
    row = {
        'scenario_id': sid,
        'reasoning_bseq_feas': bool(rec.get('reasoning_bseq_feas', rec.get('reasoning_bseq_feasible', False))),
        'reasoning_score': pd.to_numeric(rec.get('reasoning_score', np.nan), errors='coerce'),
        'scp_A': bool(scp4.get('A', False)),
        'scp_B': bool(scp4.get('B', False)),
        'scp_C': bool(scp4.get('C', False)),
        'scp_D': bool(scp4.get('D', False)),
    }
    row['scp_all4'] = row['scp_A'] and row['scp_B'] and row['scp_C'] and row['scp_D']
    scenario_rows.append(row)

    pipelines = rec.get('pipelines') or {}
    for pid in ['A', 'B', 'C', 'D']:
        p = pipelines.get(pid) or {}
        m = p.get('metrics') or {}
        pipeline_rows.append({
            'scenario_id': sid,
            'pipeline': pid,
            'scp_converged': bool(p.get('scp_converged', False)),
            'fuel_dv': float(m.get('fuel_dv', np.nan)),
            'transfer_time_sec': float(m.get('transfer_time_sec', np.nan)),
            'observation_score': float(m.get('observation_score', np.nan)),
            'safety_margin_m': float(m.get('safety_margin_m', np.nan)),
        })

scenario_df = pd.DataFrame(scenario_rows).sort_values('scenario_id').reset_index(drop=True)
pipeline_df = pd.DataFrame(pipeline_rows).sort_values(['scenario_id', 'pipeline']).reset_index(drop=True)


In [3]:

# Requested printout: counts from per-row feasibility assignments

num_bseq_feasible = int(scenario_df['reasoning_bseq_feas'].sum())
num_cases = int(len(scenario_df))

print('(1) bseq feasibility (assigned per row via scenario_df["reasoning_bseq_feas"])')
print(f'Feasible cases: {num_bseq_feasible} / {num_cases}')
print(f'B-seq feasibility rate: {num_bseq_feasible / num_cases:.4f}')
print()

scp_cols = ['scp_A', 'scp_B', 'scp_C', 'scp_D']
scp_counts = scenario_df[scp_cols].sum().astype(int)

print('(2) SCP feasibility counts (4x1 for A, B, C, D)')
print(f"A: {int(scp_counts['scp_A'])/ num_cases}")
print(f"B: {int(scp_counts['scp_B']) / num_cases}")
print(f"C: {int(scp_counts['scp_C']) / num_cases}")
print(f"D: {int(scp_counts['scp_D']) / num_cases}")



(1) bseq feasibility (assigned per row via scenario_df["reasoning_bseq_feas"])
Feasible cases: 500 / 500
B-seq feasibility rate: 1.0000

(2) SCP feasibility counts (4x1 for A, B, C, D)
A: 0.752
B: 0.68
C: 0.132
D: 0.134


In [4]:
"""
Score codes:
-2: no focused metric extracted (0 metrics)
-1: D is NaN on any focused metric
0: 1-of-1 focused metric, D is losing (not advantageous)
1: 2-of-2 focused metrics, D is losing on both
2: 2 metrics, only the first metric is losing
3: 2 metrics, only the second metric is losing
4: 1-of-1 focused metric, D is advantageous
5: 2-of-2 focused metrics, D is advantageous on both
"""




'\nScore codes:\n-2: no focused metric extracted (0 metrics)\n-1: D is NaN on any focused metric\n0: 1-of-1 focused metric, D is losing (not advantageous)\n1: 2-of-2 focused metrics, D is losing on both\n2: 2 metrics, only the first metric is losing\n3: 2 metrics, only the second metric is losing\n4: 1-of-1 focused metric, D is advantageous\n5: 2-of-2 focused metrics, D is advantageous on both\n'

In [5]:
from collections import Counter

SCORE_ORDER = [-2, -1, 0, 1, 2, 3, 4, 5]
PIPELINES = ['A', 'B', 'C', 'D']


def score_counts_by_pipeline(records, eval_key):
    counts = {pid: Counter() for pid in PIPELINES}
    for rec in records:
        score_by_pipeline = ((rec.get(eval_key) or {}).get('score_by_pipeline') or {})
        for pid in PIPELINES:
            if pid in score_by_pipeline:
                counts[pid][int(score_by_pipeline[pid])] += 1
    return counts


def format_done_line(name, counter):
    return f"[done] {name} " + " ".join(f"{k}={counter.get(k, 0)}" for k in SCORE_ORDER)

print('[done] reasoning_score_counts_by_pipeline')
reasoning_counts = score_counts_by_pipeline(records, 'reasoning_eval')
for pid in PIPELINES:
    print(format_done_line(pid, reasoning_counts[pid]))

print('[done] intent_score_counts_by_pipeline')
intent_counts = score_counts_by_pipeline(records, 'intent_eval')
for pid in PIPELINES:
    print(format_done_line(pid, intent_counts[pid]))


[done] reasoning_score_counts_by_pipeline
[done] A -2=0 -1=124 0=0 1=230 2=112 3=13 4=0 5=21
[done] B -2=0 -1=160 0=0 1=98 2=26 3=102 4=0 5=114
[done] C -2=0 -1=434 0=0 1=35 2=23 3=7 4=0 5=1
[done] D -2=0 -1=433 0=0 1=11 2=9 3=20 4=0 5=27
[done] intent_score_counts_by_pipeline
[done] A -2=0 -1=124 0=0 1=193 2=66 3=71 4=0 5=46
[done] B -2=0 -1=160 0=0 1=130 2=74 3=65 4=0 5=71
[done] C -2=0 -1=434 0=0 1=36 2=9 3=15 4=0 5=6
[done] D -2=0 -1=433 0=0 1=13 2=15 3=12 4=0 5=27


In [6]:
from collections import Counter

consistency_counts = Counter()

for rec in records:
    reasoning_metrics = set(((rec.get('reasoning_eval') or {}).get('focused_metrics') or []))
    intent_metrics = set(((rec.get('intent_eval') or {}).get('focused_metrics') or []))
    overlap = len(reasoning_metrics & intent_metrics)
    consistency_counts[min(overlap, 2)] += 1

print(
    '[done] intent_reasoning_focused_metrics_consistency '
    + ' '.join(f"{k}={consistency_counts.get(k, 0)}" for k in [2, 1, 0])
)


[done] intent_reasoning_focused_metrics_consistency 2=151 1=292 0=57
